In [ ]:
%load_ext autoreload
%autoreload 2
import dt4dds_benchmark
import plotly.express as px
import pandas as pd
import numpy as np

data = dt4dds_benchmark.analysis.Dataset.combine(*[dt4dds_benchmark.pipelines.HDF5Manager(f'./data/{w}/{s}.hdf5').get_data() for s in (
   'aeon_low', 'aeon_medium', 'aeon_high', 'rs_low', 'rs_medium', 'rs_high', 'modulation_medium', 'ldpc_medium', 'dbgps_low', 'dbgps_medium', 'dbgps_high',
) for w in (
    'insertion',
    'deletion',
    'substitution',
    # 'dropout',
)])

### get the threshold values by codec, clustering, and scenario

In [ ]:
df = data.get_fits_by_group(['codec.type', 'codec.name', 'clustering.name', 'clustering.type', 'metadata.name'], on='workflow.overall_rate', additional_agg={'code_rate': 'mean'})
df['code_rate'] = df['code_rate'].map('{:.2f}'.format)

df

### plot substitution, deletion, and insertion

In [ ]:
plotdf = df.loc[df['metadata.name'].isin(['substitution', 'deletion', 'insertion'])].copy()
plotdf['id'] = plotdf['codec.type'] + '_' + plotdf['codec.name'] + '_' + plotdf['clustering.type']
plotdf['codec.type'] = plotdf['codec.type'].str.replace('Modulation', 'Mod').str.replace('DBGPS', 'EDBGPS')
plotdf = plotdf.loc[plotdf['clustering.type'] != 'BasicSet'].copy()

plotdf

In [ ]:
fig = dt4dds_benchmark.analysis.plotting.tiered_bar(
    plotdf.sort_values(['codec.type', 'code_rate', 'metadata.name']),
    "codec.type",
    "code_rate",
    "threshold",
    color_by = "metadata.name",
    # error_upper = "threshold_50%",
    # error_lower = "threshold_99%",
    color_discrete_map={'substitution': '#e6550d', 'deletion': '#3182bd', 'insertion': '#2ca25f'},
)
fig.update_yaxes(
    title_text='Error rate per nt',
    tickformat=",.0%",
    range=[0, 0.15],
    # type="log",
    dtick=0.05,
)
fig.update_layout(
    width=34,
    height=140,
    margin=dict(l=0, r=2, t=2, b=30),
    showlegend=False,
)


fig = dt4dds_benchmark.analysis.plotting.standardize_plot(fig)
fig.update_xaxes(
    tickfont_size=28/3, 
    tickangle=0,
)
fig.show()
fig.write_image(f'./figures/individual_plot.svg')

# export data
plotdf.to_csv('./figures/individual_plot.csv', index=False)